In [2]:
import ee
import pandas as pd
import geopandas as gpd
from datetime import datetime, timedelta
import time


# Initialize Earth Engine
try:
    ee.Initialize()
except Exception as e:
    print("Earth Engine initialization error. Make sure you're authenticated:")
    print("Run 'earthengine authenticate' in your terminal or use ee.Authenticate()")
    print(f"Error details: {e}")


In [3]:
start_date = '2021-03-01'
end_date = '2021-03-31'

In [4]:
def sample_gee_data_with_gdf(
    gdf,
    image_collection_name,
    start_date,
    end_date,
    bands=None,
    scale=1000,
    crs='EPSG:4326',
    reducer=None,
    filter_params=None,
    temporal_resolution='day',
    suffix=None,
    verbose=True
):
    """
    Sample data from Google Earth Engine image collection over a time period
    using points from a GeoDataFrame and append results back to the GeoDataFrame.
    
    Parameters:
    -----------
    gdf : geopandas.GeoDataFrame
        GeoDataFrame containing the points to sample
    
    image_collection_name : str
        Name of the GEE image collection (e.g., 'MODIS/006/MOD13Q1')
    
    start_date : str
        Start date in 'YYYY-MM-DD' format
    
    end_date : str
        End date in 'YYYY-MM-DD' format
    
    bands : list, optional
        List of band names to include in the sampling
    
    scale : int, optional
        Scale in meters for sampling (pixel size)
    
    crs : str, optional
        Coordinate reference system code
    
    reducer : ee.Reducer, optional
        Reducer to use for aggregation. Default is ee.Reducer.mean()
    
    filter_params : dict, optional
        Additional filtering parameters for the image collection
    
    temporal_resolution : str, optional
        Time step for sampling: 'day', 'week', 'month', or 'year'
    
    suffix : str, optional
        Suffix to add to the column names of the sampled data
        
    verbose : bool, optional
        Whether to print progress information
    
    Returns:
    --------
    geopandas.GeoDataFrame
        Original GeoDataFrame with added columns for the sampled data
    """
    if verbose:
        print(f"Setting up sampling for {image_collection_name}")
    
    # Ensure GDF is in EPSG:4326 for compatibility with Earth Engine
    original_crs = gdf.crs
    if gdf.crs != "EPSG:4326":
        gdf_for_ee = gdf.to_crs("EPSG:4326")
    else:
        gdf_for_ee = gdf.copy()
    
    # Convert GeoDataFrame points to Earth Engine FeatureCollection
    features = []
    for idx, row in gdf_for_ee.iterrows():
        point = ee.Geometry.Point([row.geometry.x, row.geometry.y])
        # Add index as a property to match results back to GDF
        feature = ee.Feature(point, {'point_idx': idx})
        features.append(feature)
    
    ee_points = ee.FeatureCollection(features)
    
    # Convert dates to GEE format
    start = ee.Date(start_date)
    end = ee.Date(end_date)
    
    # Set default reducer if none provided
    if reducer is None:
        reducer = ee.Reducer.mean()
    
    # Get the image collection
    collection = ee.ImageCollection(image_collection_name)
    
    # Apply date filter
    collection = collection.filterDate(start, end)
    
    # Apply additional filters if provided
    if filter_params:
        for param, value in filter_params.items():
            if param == 'filterBounds':
                collection = collection.filterBounds(value)
            elif param == 'filter':
                collection = collection.filter(value)
            elif param == 'sort':
                collection = collection.sort(value)
            # Add more filter types as needed
    
    # Select bands if specified
    if bands:
        collection = collection.select(bands)
    
    # Determine time steps based on temporal_resolution
    time_delta = {
        'day': timedelta(days=1),
        'week': timedelta(weeks=1),
        'month': timedelta(days=30),
        'year': timedelta(days=365)
    }
    
    if temporal_resolution not in time_delta:
        raise ValueError(f"Invalid temporal_resolution: {temporal_resolution}. "
                         f"Choose from {list(time_delta.keys())}")
    
    delta = time_delta[temporal_resolution]
    
    # Initialize results storage
    # Dictionary with point_idx as key and list of time-value pairs as values
    results_by_point = {idx: [] for idx in gdf.index}
    
    # Calculate total number of time steps for progress reporting
    start_dt = datetime.strptime(start_date, '%Y-%m-%d')
    end_dt = datetime.strptime(end_date, '%Y-%m-%d')
    total_steps = max(1, int((end_dt - start_dt).days / delta.days))
    
    # Process each time step
    current_date = start_dt
    step_count = 0
    
    collected_dates = []
    
    while current_date <= end_dt:
        step_count += 1
        
        # Format the current date and next date
        current_date_str = current_date.strftime('%Y-%m-%d')
        next_date = current_date + delta
        next_date_str = next_date.strftime('%Y-%m-%d')
        
        if verbose:
            print(f"Processing time step {step_count}/{total_steps}: {current_date_str} to {next_date_str}")
        
        # Filter collection for the current time step
        step_collection = collection.filterDate(current_date_str, next_date_str)
        
        # Skip if no images available for this time step
        image_count = step_collection.size().getInfo()
        if image_count == 0:
            if verbose:
                print(f"  No images found for period {current_date_str} to {next_date_str}, skipping...")
            current_date = next_date
            continue
        
        # Get the median/mosaic image for this time period
        if image_count == 1:
            image = step_collection.first()
        else:
            image = step_collection.median()  # or .mosaic() depending on your needs
        
        try:
            # Sample the image at the points
            samples = image.sampleRegions(
                collection=ee_points,
                properties=['point_idx'],
                scale=scale,
                geometries=True,
                tileScale=4  # Helps with computational limits
            )
            
            # Get the results
            try:
                sample_data = samples.getInfo()
                
                # Process each sample
                for feature in sample_data['features']:
                    point_idx = feature['properties']['point_idx']
                    sample_values = {k: v for k, v in feature['properties'].items() if k != 'point_idx'}
                    
                    # Add date information to the sample values
                    sample_values['date'] = current_date_str
                    
                    # Store in the results dictionary
                    results_by_point[point_idx].append(sample_values)
                
                collected_dates.append(current_date_str)
                
                if verbose:
                    print(f"  Successfully sampled {len(sample_data['features'])} points")
            
            except ee.ee_exception.EEException as e:
                print(f"  Error getting sample data: {e}")
                # Implement exponential backoff if needed for rate limits
                time.sleep(1)
        
        except Exception as e:
            print(f"  Error sampling image: {e}")
        
        # Move to next time step
        current_date = next_date
    
    if not collected_dates:
        print("No data was collected. Check your parameters and image availability.")
        return gdf
    
    # Process the results and add to the GeoDataFrame
    results_dfs = []
    
    # Decide on column naming strategy based on temporal resolution
    if temporal_resolution == 'day' or len(collected_dates) == 1:
        # For daily data or single date, add date to column name
        for point_idx, samples in results_by_point.items():
            if not samples:
                continue
                
            point_df = pd.DataFrame(samples)
            # Convert date to datetime
            point_df['date'] = pd.to_datetime(point_df['date'])
            
            # Pivot the data to have dates as columns
            for band in point_df.columns:
                if band != 'date':
                    for date_str in point_df['date'].dt.strftime('%Y%m%d').unique():
                        date_data = point_df[point_df['date'].dt.strftime('%Y%m%d') == date_str]
                        if not date_data.empty:
                            col_name = f"{band}_{date_str}"
                            if suffix:
                                col_name = f"{col_name}_{suffix}"
                            gdf.loc[point_idx, col_name] = date_data[band].values[0]
    else:
        # For coarser temporal resolution, calculate statistics over the time period
        for point_idx, samples in results_by_point.items():
            if not samples:
                continue
                
            point_df = pd.DataFrame(samples)
            
            # Calculate statistics for each band
            for band in point_df.columns:
                if band != 'date':
                    # Mean value over the time period
                    mean_col = f"{band}_mean"
                    if suffix:
                        mean_col = f"{mean_col}_{suffix}"
                    gdf.loc[point_idx, mean_col] = point_df[band].mean()
                    
                    # Min value over the time period
                    min_col = f"{band}_min"
                    if suffix:
                        min_col = f"{min_col}_{suffix}"
                    gdf.loc[point_idx, min_col] = point_df[band].min()
                    
                    # Max value over the time period
                    max_col = f"{band}_max"
                    if suffix:
                        max_col = f"{max_col}_{suffix}"
                    gdf.loc[point_idx, max_col] = point_df[band].max()
                    
                    # Standard deviation over the time period
                    std_col = f"{band}_std"
                    if suffix:
                        std_col = f"{std_col}_{suffix}"
                    gdf.loc[point_idx, std_col] = point_df[band].std()
    
    if verbose:
        print(f"Sampling complete. Added {len(gdf.columns) - len(gdf_for_ee.columns)} new columns to the GeoDataFrame.")
    
    return gdf

In [5]:

# Example usage:
import geopandas as gpd
from shapely.geometry import Point

gdf = gpd.read_file('./data/soilpoints.shp')


In [6]:

# Sample NDVI data and add to the GeoDataFrame
gdf_with_ndvi = sample_gee_data_with_gdf(
    gdf=gdf,
    image_collection_name='MODIS/006/MOD13Q1', 
    start_date='2020-01-01',
    end_date='2020-12-31',
    bands=['NDVI'],
    temporal_resolution='month',
    suffix='modis'
)

# Sample precipitation data and add to the GeoDataFrame
gdf_with_ndvi_and_precip = sample_gee_data_with_gdf(
    gdf=gdf_with_ndvi,
    image_collection_name='NASA/GPM_L3/IMERG_V06',
    start_date='2020-01-01',
    end_date='2020-03-31',
    bands=['precipitationCal'],
    temporal_resolution='month',
    suffix='gpm'
)

Setting up sampling for MODIS/006/MOD13Q1
Processing time step 1/12: 2020-01-01 to 2020-01-31


c:\Users\roywaswa\AppData\Roaming\mamba\envs\dsa\Lib\site-packages\ee\deprecation.py:207: DeprecationWarning: 

Attention required for MODIS/006/MOD13Q1! You are using a deprecated asset.
To make sure your code keeps working, please update it.
Learn more: https://developers.google.com/earth-engine/datasets/catalog/MODIS_006_MOD13Q1

  warnings.warn(warning, category=DeprecationWarning)


  Successfully sampled 159 points
Processing time step 2/12: 2020-01-31 to 2020-03-01
  Successfully sampled 159 points
Processing time step 3/12: 2020-03-01 to 2020-03-31
  Successfully sampled 159 points
Processing time step 4/12: 2020-03-31 to 2020-04-30
  Successfully sampled 159 points
Processing time step 5/12: 2020-04-30 to 2020-05-30
  Successfully sampled 159 points
Processing time step 6/12: 2020-05-30 to 2020-06-29
  Successfully sampled 159 points
Processing time step 7/12: 2020-06-29 to 2020-07-29
  Successfully sampled 159 points
Processing time step 8/12: 2020-07-29 to 2020-08-28
  Successfully sampled 159 points
Processing time step 9/12: 2020-08-28 to 2020-09-27
  Successfully sampled 159 points
Processing time step 10/12: 2020-09-27 to 2020-10-27
  Successfully sampled 159 points
Processing time step 11/12: 2020-10-27 to 2020-11-26
  Successfully sampled 159 points
Processing time step 12/12: 2020-11-26 to 2020-12-26
  Successfully sampled 159 points
Processing time s

c:\Users\roywaswa\AppData\Roaming\mamba\envs\dsa\Lib\site-packages\ee\deprecation.py:207: DeprecationWarning: 

Attention required for NASA/GPM_L3/IMERG_V06! You are using a deprecated asset.
To make sure your code keeps working, please update it.
Learn more: https://developers.google.com/earth-engine/datasets/catalog/NASA_GPM_L3_IMERG_V06

  warnings.warn(warning, category=DeprecationWarning)


  Successfully sampled 159 points
Processing time step 2/3: 2020-01-31 to 2020-03-01
  Successfully sampled 159 points
Processing time step 3/3: 2020-03-01 to 2020-03-31
  Successfully sampled 159 points
Processing time step 4/3: 2020-03-31 to 2020-04-30
  No images found for period 2020-03-31 to 2020-04-30, skipping...
Sampling complete. Added 4 new columns to the GeoDataFrame.


In [7]:
gdf_with_ndvi_and_precip.columns

Index(['B11', 'B12', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9',
       'BSI', 'CPR', 'NBR', 'NDI', 'NDSI', 'NDVI', 'RVI', 'SAVI', 'VH', 'VV',
       'VVVHR', 'predSOC', 'geometry', 'NDVI_mean_modis', 'NDVI_min_modis',
       'NDVI_max_modis', 'NDVI_std_modis', 'precipitationCal_mean_gpm',
       'precipitationCal_min_gpm', 'precipitationCal_max_gpm',
       'precipitationCal_std_gpm'],
      dtype='object')

In [8]:
def sample_all_datasets(gdf, start_date, end_date, verbose=True):
    """
    Sample all specified datasets and add results to the provided GeoDataFrame.
    
    Parameters:
    -----------
    gdf : geopandas.GeoDataFrame
        GeoDataFrame containing the points to sample
    
    start_date : str
        Start date in 'YYYY-MM-DD' format
    
    end_date : str
        End date in 'YYYY-MM-DD' format
        
    verbose : bool
        Whether to print progress information
        
    Returns:
    --------
    geopandas.GeoDataFrame
        Original GeoDataFrame with added columns for all sampled datasets
    """
    if verbose:
        print(f"Beginning sampling process for all datasets from {start_date} to {end_date}")
        print(f"Processing {len(gdf)} points")
    
    # Make a copy of the input GeoDataFrame to avoid modifying the original
    result_gdf = gdf.copy()
    
    # Start timing
    total_start_time = time.time()
    
    # Track datasets that fail to sample
    failed_datasets = []
    
    # 1. OPTICAL IMAGERY
    
    # 1.1 Sentinel-2
    try:
        if verbose:
            print("\n=== Sampling Sentinel-2 ===")
        
        # Define key bands for Sentinel-2
        s2_bands = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'SCL']
        
        result_gdf = sample_gee_data_with_gdf(
            gdf=result_gdf,
            image_collection_name='COPERNICUS/S2_SR_HARMONIZED',
            start_date=start_date,
            end_date=end_date,
            bands=s2_bands,
            scale=10,  # Using highest resolution
            temporal_resolution='month',  # Monthly composites
            suffix='s2',
            verbose=verbose
        )
        
        # Calculate NDVI and add as derived product (if B8 and B4 were successfully sampled)
        ndvi_cols = [col for col in result_gdf.columns if '_B8_mean_s2' in col]
        red_cols = [col for col in result_gdf.columns if '_B4_mean_s2' in col]
        
        if ndvi_cols and red_cols:
            for nir_col, red_col in zip(ndvi_cols, red_cols):
                # Extract date part from column name
                suffix = nir_col.replace('_B8_mean_s2', '')
                # Create NDVI column
                ndvi_col = f"{suffix}_NDVI_s2"
                result_gdf[ndvi_col] = (result_gdf[nir_col] - result_gdf[red_col]) / (result_gdf[nir_col] + result_gdf[red_col])
    except Exception as e:
        print(f"Failed to sample Sentinel-2 data: {e}")
        failed_datasets.append("Sentinel-2")
    
    # 1.2 Landsat 8
    try:
        if verbose:
            print("\n=== Sampling Landsat 8 ===")
        
        landsat_bands = ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7', 'QA_PIXEL']
        
        result_gdf = sample_gee_data_with_gdf(
            gdf=result_gdf,
            image_collection_name='LANDSAT/LC08/C02/T1_L2',
            start_date=start_date,
            end_date=end_date,
            bands=landsat_bands,
            scale=30,
            temporal_resolution='month',
            suffix='ls8',
            verbose=verbose
        )
        
        # Calculate NDVI for Landsat 8 (NIR = B5, Red = B4)
        nir_cols = [col for col in result_gdf.columns if '_SR_B5_mean_ls8' in col]
        red_cols = [col for col in result_gdf.columns if '_SR_B4_mean_ls8' in col]
        
        if nir_cols and red_cols:
            for nir_col, red_col in zip(nir_cols, red_cols):
                # Extract date part from column name
                suffix = nir_col.replace('_SR_B5_mean_ls8', '')
                # Create NDVI column
                ndvi_col = f"{suffix}_NDVI_ls8"
                result_gdf[ndvi_col] = (result_gdf[nir_col] - result_gdf[red_col]) / (result_gdf[nir_col] + result_gdf[red_col])
    except Exception as e:
        print(f"Failed to sample Landsat 8 data: {e}")
        failed_datasets.append("Landsat 8")
        
    # 1.3 MODIS
    try:
        if verbose:
            print("\n=== Sampling MODIS Vegetation Indices ===")
        
        result_gdf = sample_gee_data_with_gdf(
            gdf=result_gdf,
            image_collection_name='MODIS/006/MOD13Q1',
            start_date=start_date,
            end_date=end_date,
            bands=['NDVI', 'EVI', 'DetailedQA'],
            scale=250,
            temporal_resolution='month',
            suffix='modis',
            verbose=verbose
        )
    except Exception as e:
        print(f"Failed to sample MODIS data: {e}")
        failed_datasets.append("MODIS")
    
    # 2. RADAR IMAGERY
    
    # 2.1 Sentinel-1
    try:
        if verbose:
            print("\n=== Sampling Sentinel-1 ===")
        
        result_gdf = sample_gee_data_with_gdf(
            gdf=result_gdf,
            image_collection_name='COPERNICUS/S1_GRD',
            start_date=start_date,
            end_date=end_date,
            bands=['VV', 'VH'],
            scale=10,
            temporal_resolution='month',
            suffix='s1',
            filter_params={'filter': ee.Filter.eq('instrumentMode', 'IW')},
            verbose=verbose
        )
        
        # Calculate VH/VV ratio (useful for soil moisture estimation)
        vh_cols = [col for col in result_gdf.columns if '_VH_mean_s1' in col]
        vv_cols = [col for col in result_gdf.columns if '_VV_mean_s1' in col]
        
        if vh_cols and vv_cols:
            for vh_col, vv_col in zip(vh_cols, vv_cols):
                # Extract date part from column name
                suffix = vh_col.replace('_VH_mean_s1', '')
                # Create ratio column
                ratio_col = f"{suffix}_VH_VV_ratio_s1"
                result_gdf[ratio_col] = result_gdf[vh_col] / result_gdf[vv_col]
    except Exception as e:
        print(f"Failed to sample Sentinel-1 data: {e}")
        failed_datasets.append("Sentinel-1")
    
    # 3. CLIMATE DATA
    
    # 3.1 CHIRPS
    try:
        if verbose:
            print("\n=== Sampling CHIRPS Precipitation ===")
        
        result_gdf = sample_gee_data_with_gdf(
            gdf=result_gdf,
            image_collection_name='UCSB-CHG/CHIRPS/PENTAD',
            start_date=start_date,
            end_date=end_date,
            bands=['precipitation'],
            scale=5000,
            temporal_resolution='month',
            suffix='chirps',
            verbose=verbose
        )
    except Exception as e:
        print(f"Failed to sample CHIRPS data: {e}")
        failed_datasets.append("CHIRPS")
    
    # 3.2 TerraClimate
    try:
        if verbose:
            print("\n=== Sampling TerraClimate ===")
        
        terraclimate_bands = ['pr', 'tmmx', 'tmmn', 'pdsi', 'pet', 'soil']
        
        result_gdf = sample_gee_data_with_gdf(
            gdf=result_gdf,
            image_collection_name='IDAHO_EPSCOR/TERRACLIMATE',
            start_date=start_date,
            end_date=end_date,
            bands=terraclimate_bands,
            scale=4000,
            temporal_resolution='month',
            suffix='terraclim',
            verbose=verbose
        )
    except Exception as e:
        print(f"Failed to sample TerraClimate data: {e}")
        failed_datasets.append("TerraClimate")
    
    # 4. TOPOGRAPHIC DATA
    
    # 4.1 SRTM DEM (static data, use first date only)
    try:
        if verbose:
            print("\n=== Sampling SRTM DEM ===")
        
        # For static data like elevation, just use a single date
        srtm_date = start_date
        
        # Create an ee.Image from the SRTM dataset
        srtm = ee.Image('USGS/SRTMGL1_003')
        
        # Sample single date
        temp_gdf = sample_gee_data_with_gdf(
            gdf=result_gdf,
            image_collection_name='USGS/SRTMGL1_003',
            start_date=srtm_date,
            end_date=srtm_date,
            bands=['elevation'],
            scale=30,
            temporal_resolution='day',
            suffix='srtm',
            verbose=verbose
        )
        
        # Copy only the elevation columns to result_gdf
        elev_cols = [col for col in temp_gdf.columns if 'elevation' in col and 'srtm' in col]
        for col in elev_cols:
            result_gdf[col.replace('_20', '_')] = temp_gdf[col]  # Remove date from column name
    except Exception as e:
        print(f"Failed to sample SRTM data: {e}")
        failed_datasets.append("SRTM")
    
    # 4.2 ALOS World 3D
    try:
        if verbose:
            print("\n=== Sampling ALOS DEM ===")
        
        # For static data like elevation, just use a single date
        alos_date = start_date
        
        temp_gdf = sample_gee_data_with_gdf(
            gdf=result_gdf,
            image_collection_name='JAXA/ALOS/AW3D30/V2_2',
            start_date=alos_date,
            end_date=alos_date,
            bands=['AVE_DSM'],
            scale=30,
            temporal_resolution='day',
            suffix='alos',
            verbose=verbose
        )
        
        # Copy only the elevation columns to result_gdf
        elev_cols = [col for col in temp_gdf.columns if 'AVE_DSM' in col and 'alos' in col]
        for col in elev_cols:
            result_gdf[col.replace('_20', '_')] = temp_gdf[col]  # Remove date from column name
    except Exception as e:
        print(f"Failed to sample ALOS data: {e}")
        failed_datasets.append("ALOS")
    
    # 5. SOIL DATA
    
    # 5.1 SoilGrids (using OpenLandMap soil organic carbon)
    try:
        if verbose:
            print("\n=== Sampling SoilGrids Organic Carbon ===")
        
        # SoilGrids is static data
        soil_date = start_date
        
        # Use OpenLandMap soil organic carbon content
        temp_gdf = sample_gee_data_with_gdf(
            gdf=result_gdf,
            image_collection_name='OpenLandMap/SOL/SOL_ORGANIC-CARBON_USDA-6A1C_M/v02',
            start_date=soil_date,
            end_date=soil_date,
            bands=['b0', 'b10', 'b30', 'b60', 'b100', 'b200'],  # Different soil depths (cm)
            scale=250,
            temporal_resolution='day',
            suffix='soc',
            verbose=verbose
        )
        
        # Copy soil data columns to result_gdf and rename for clarity
        depth_mapping = {
            'b0': 'SOC_0cm',
            'b10': 'SOC_10cm',
            'b30': 'SOC_30cm',
            'b60': 'SOC_60cm',
            'b100': 'SOC_100cm', 
            'b200': 'SOC_200cm'
        }
        
        for old_prefix, new_name in depth_mapping.items():
            cols = [col for col in temp_gdf.columns if old_prefix in col and 'soc' in col]
            for col in cols:
                new_col = col.replace(old_prefix, new_name).replace('_20', '_')
                result_gdf[new_col] = temp_gdf[col]
    except Exception as e:
        print(f"Failed to sample SoilGrids data: {e}")
        failed_datasets.append("SoilGrids")
    
    # 6. OTHER DATA
    
    # 6.1 SMAP Soil Moisture
    try:
        if verbose:
            print("\n=== Sampling SMAP Soil Moisture ===")
        
        result_gdf = sample_gee_data_with_gdf(
            gdf=result_gdf,
            image_collection_name='NASA_USDA/HSL/SMAP_soil_moisture',
            start_date=start_date,
            end_date=end_date,
            bands=['ssm', 'susm'],  # Surface and subsurface soil moisture
            scale=1000,
            temporal_resolution='month',
            suffix='smap',
            verbose=verbose
        )
    except Exception as e:
        print(f"Failed to sample SMAP data: {e}")
        failed_datasets.append("SMAP")
    
    # Calculate total time
    total_time = time.time() - total_start_time
    hours, remainder = divmod(total_time, 3600)
    minutes, seconds = divmod(remainder, 60)
    
    if verbose:
        print("\n=== Sampling Complete ===")
        print(f"Total sampling time: {int(hours)}:{int(minutes):02d}:{seconds:.2f}")
        print(f"Added {len(result_gdf.columns) - len(gdf.columns)} new columns to the GeoDataFrame")
        
        if failed_datasets:
            print(f"Warning: Failed to sample {len(failed_datasets)} datasets: {', '.join(failed_datasets)}")
    
    return result_gdf

In [9]:

# Either load your existing GeoDataFrame with points
my_points_gdf = gdf[['predSOC','geometry']].copy()
type(my_points_gdf)

geopandas.geodataframe.GeoDataFrame

In [10]:
# # Sample all datasets
result_gdf = sample_all_datasets(my_points_gdf, start_date, end_date)
result_gdf
# # Save the results
# # result_gdf.to_file('points_with_earth_engine_data.shp')
# # Or if there are too many columns for a shapefile
# # result_gdf.to_file('points_with_earth_engine_data.gpkg', driver='GPKG')

Beginning sampling process for all datasets from 2021-03-01 to 2021-03-31
Processing 159 points

=== Sampling Sentinel-2 ===
Setting up sampling for COPERNICUS/S2_SR_HARMONIZED


Processing time step 1/1: 2021-03-01 to 2021-03-31
  Successfully sampled 159 points
Processing time step 2/1: 2021-03-31 to 2021-04-30
  No images found for period 2021-03-31 to 2021-04-30, skipping...
Sampling complete. Added 7 new columns to the GeoDataFrame.

=== Sampling Landsat 8 ===
Setting up sampling for LANDSAT/LC08/C02/T1_L2
Processing time step 1/1: 2021-03-01 to 2021-03-31
  Successfully sampled 159 points
Processing time step 2/1: 2021-03-31 to 2021-04-30
  No images found for period 2021-03-31 to 2021-04-30, skipping...
Sampling complete. Added 7 new columns to the GeoDataFrame.

=== Sampling MODIS Vegetation Indices ===
Setting up sampling for MODIS/006/MOD13Q1
Processing time step 1/1: 2021-03-01 to 2021-03-31
  Successfully sampled 159 points
Processing time step 2/1: 2021-03-31 to 2021-04-30
  No images found for period 2021-03-31 to 2021-04-30, skipping...
Sampling complete. Added 3 new columns to the GeoDataFrame.

=== Sampling Sentinel-1 ===
Setting up sampling fo

c:\Users\roywaswa\AppData\Roaming\mamba\envs\dsa\Lib\site-packages\ee\deprecation.py:207: DeprecationWarning: 

Attention required for JAXA/ALOS/AW3D30/V2_2! You are using a deprecated asset.
To make sure your code keeps working, please update it.
Learn more: https://developers.google.com/earth-engine/datasets/catalog/JAXA_ALOS_AW3D30_V2_2

  warnings.warn(warning, category=DeprecationWarning)


Failed to sample ALOS data: ImageCollection.load: Expected asset 'JAXA/ALOS/AW3D30/V2_2' to be an ImageCollection, found 'Image'.

=== Sampling SoilGrids Organic Carbon ===
Setting up sampling for OpenLandMap/SOL/SOL_ORGANIC-CARBON_USDA-6A1C_M/v02
Processing time step 1/1: 2021-03-01 to 2021-03-02
Failed to sample SoilGrids data: ImageCollection.load: Expected asset 'OpenLandMap/SOL/SOL_ORGANIC-CARBON_USDA-6A1C_M/v02' to be an ImageCollection, found 'Image'.

=== Sampling SMAP Soil Moisture ===
Setting up sampling for NASA_USDA/HSL/SMAP_soil_moisture
Processing time step 1/1: 2021-03-01 to 2021-03-31


c:\Users\roywaswa\AppData\Roaming\mamba\envs\dsa\Lib\site-packages\ee\deprecation.py:207: DeprecationWarning: 

Attention required for NASA_USDA/HSL/SMAP_soil_moisture! You are using a deprecated asset.
To make sure your code keeps working, please update it.
Learn more: https://developers.google.com/earth-engine/datasets/catalog/NASA_USDA_HSL_SMAP_soil_moisture

  warnings.warn(warning, category=DeprecationWarning)


  No images found for period 2021-03-01 to 2021-03-31, skipping...
Processing time step 2/1: 2021-03-31 to 2021-04-30
  No images found for period 2021-03-31 to 2021-04-30, skipping...
No data was collected. Check your parameters and image availability.

=== Sampling Complete ===
Total sampling time: 0:00:50.06
Added 24 new columns to the GeoDataFrame


,predSOC,geometry,B11_20210301_s2,B12_20210301_s2,B2_20210301_s2,B3_20210301_s2,B4_20210301_s2,B8_20210301_s2,SCL_20210301_s2,QA_PIXEL_20210301_ls8,...,DetailedQA_20210301_modis,EVI_20210301_modis,NDVI_20210301_modis,precipitation_20210301_chirps,pdsi_20210301_terraclim,pet_20210301_terraclim,pr_20210301_terraclim,soil_20210301_terraclim,tmmn_20210301_terraclim,tmmx_20210301_terraclim
0,9.9325,POINT (34.64061 0.71142),3025.0,2184.0,774.5,1059.0,1189.5,2593.0,4.5,22052.0,...,2150.5,3197.5,5324.5,9.328049,176.0,1537.0,66.0,225.0,141.0,304.0
1,12.5035,POINT (34.64034 0.71367),3194.5,2541.0,866.0,1111.5,1336.0,2719.5,4.5,22825.0,...,2150.5,3135.5,5325.5,9.328049,176.0,1537.0,66.0,225.0,141.0,304.0
2,9.4790,POINT (34.64052 0.716),3003.0,2394.5,996.0,1289.5,1473.0,2891.0,5.5,22856.0,...,2150.5,3123.5,5335.5,9.328049,176.0,1537.0,66.0,225.0,141.0,304.0
3,16.8675,POINT (34.64151 0.71924),2943.0,2152.0,792.0,1069.0,1099.0,2751.0,5.5,22052.0,...,2150.5,3051.5,5230.5,8.940701,176.0,1537.0,66.0,225.0,141.0,304.0
4,15.0005,POINT (34.64277 0.71924),3001.5,2095.5,816.0,1068.5,1078.0,2827.0,5.5,22856.0,...,2150.5,3051.5,5230.5,8.940701,176.0,1537.0,66.0,225.0,141.0,304.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
154,19.2500,POINT (34.71328 0.71223),2869.5,2338.0,524.0,809.5,1224.0,2033.0,5.0,21824.0,...,2185.0,2887.5,4920.0,9.328049,200.0,1538.0,60.0,235.0,133.0,302.0
155,14.3065,POINT (34.71158 0.71295),2717.0,1797.0,566.5,894.0,918.5,2826.0,4.0,21824.0,...,2185.0,3209.5,5283.5,9.328049,200.0,1538.0,60.0,235.0,133.0,302.0
156,9.9895,POINT (34.71238 0.71564),3207.0,2341.0,683.0,1024.5,1313.0,2715.0,5.0,21824.0,...,2185.0,3353.5,5398.0,9.328049,200.0,1538.0,60.0,235.0,133.0,302.0
157,10.1505,POINT (34.7096 0.71492),3135.5,2493.5,689.5,974.5,1309.0,2167.0,5.0,21824.0,...,2185.0,3394.0,5436.0,9.328049,200.0,1538.0,60.0,235.0,133.0,302.0


In [12]:
result_gdf.describe().to_csv('points_with_earth_engine_data.csv')